# 15 — Sinal de risco de churn

Este módulo classifica o sinal como `baixo`, `medio` ou `alto`. A saída apoia priorização humana e não confirma cancelamento futuro.

## MiniLM zero-shot

No caminho completo, um MiniLM multilíngue compara três hipóteses em português. O score é probabilidade do classificador NLI, não uma probabilidade calibrada de churn no negócio.

In [ ]:
CHURN_MODEL_NAME = "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"

CHURN_HYPOTHESES = {
    "baixo": "baixo risco de cancelar ou trocar o fornecedor",
    "medio": "risco moderado de cancelar ou trocar o fornecedor",
    "alto": "alto risco de cancelar ou trocar o fornecedor",
}

_CHURN_CLASSIFIER = None

_CHURN_LOAD_ERROR = None


In [ ]:
def _load_churn_classifier():
    global _CHURN_CLASSIFIER, _CHURN_LOAD_ERROR
    if _CHURN_CLASSIFIER is not None:
        return _CHURN_CLASSIFIER
    if _CHURN_LOAD_ERROR is not None:
        raise RuntimeError("MiniLM NLI indisponível.") from _CHURN_LOAD_ERROR
    try:
        from transformers import pipeline

        _CHURN_CLASSIFIER = pipeline("zero-shot-classification", model=CHURN_MODEL_NAME)
        return _CHURN_CLASSIFIER
    except Exception as error:
        _CHURN_LOAD_ERROR = error
        raise RuntimeError("MiniLM NLI indisponível.") from error

def _model_churn(transcription: str) -> dict[str, Any]:
    classifier = _load_churn_classifier()
    descriptions = list(CHURN_HYPOTHESES.values())
    reverse_labels = {description: label for label, description in CHURN_HYPOTHESES.items()}
    scores_by_label = {label: 0.0 for label in CHURN_HYPOTHESES}
    for chunk in _word_chunks(transcription):
        output = classifier(
            chunk,
            candidate_labels=descriptions,
            hypothesis_template="Esta transcrição indica {}.",
            multi_label=False,
        )
        labels = output.get("labels", [])
        scores = output.get("scores", [])
        if len(labels) != len(scores) or not labels:
            raise ValueError("Saída inválida do classificador de churn.")
        for description, score in zip(labels, scores):
            canonical = reverse_labels.get(description)
            if canonical is not None:
                scores_by_label[canonical] = max(scores_by_label[canonical], float(score))
    winner = max(scores_by_label, key=scores_by_label.get)
    return {
        "label": winner,
        "score": round(scores_by_label[winner], 6),
        "score_type": "model_probability",
        "engine": "zero_shot_nli",
        "model": CHURN_MODEL_NAME,
    }


## Sinais lexicais auditáveis

Cancelamento, não renovação, troca de fornecedor e concorrentes pesam mais; renovação e continuidade reduzem a leitura de risco quando não estão negadas.

In [ ]:
HIGH_CHURN_SIGNALS = {
    "cancelar": 3.0, "rescindir": 3.0, "encerrar o contrato": 3.0,
    "nao renovar": 3.0, "nao vamos renovar": 3.0,
    "trocar de fornecedor": 2.5, "migrar": 2.0,
    "concorrente": 1.5, "sair da totvs": 3.0,
}

MEDIUM_CHURN_SIGNALS = {
    "insatisfeito": 1.5, "insatisfeita": 1.5, "problema recorrente": 1.5,
    "problema": 1.0, "avaliando alternativas": 1.5,
    "reduzir o uso": 1.5, "preco alto": 1.0, "reclamacao": 1.0,
}

RETENTION_SIGNALS = {
    "renovar": 2.0, "continuar": 1.0, "satisfeito": 1.0,
    "satisfeita": 1.0, "expansao": 1.0,
}

def _lexical_churn(transcription: str) -> dict[str, Any]:
    normalized = f" {_normalize(transcription)} "
    high_score = sum(
        weight for phrase, weight in HIGH_CHURN_SIGNALS.items() if f" {phrase} " in normalized
    )
    medium_score = sum(
        weight for phrase, weight in MEDIUM_CHURN_SIGNALS.items() if f" {phrase} " in normalized
    )
    retention_score = 0.0
    for phrase, weight in RETENTION_SIGNALS.items():
        negated = f" nao {phrase} " in normalized or f" nao vamos {phrase} " in normalized
        if not negated and f" {phrase} " in normalized:
            retention_score += weight

    risk_score = high_score + medium_score
    if high_score >= 3.0 or risk_score >= 4.0:
        label = "alto"
        score = min(0.55 + risk_score / 10.0, 1.0)
    elif risk_score >= 1.5:
        label = "medio"
        score = min(0.35 + risk_score / 10.0, 0.74)
    else:
        label = "baixo"
        score = min(0.3 + retention_score / 10.0, 0.8)
    return {
        "label": label,
        "score": round(score, 6),
        "score_type": "heuristic",
        "engine": "lexical_churn",
        "model": None,
    }


## Roteamento por modo

A função de roteamento segue a mesma política transparente do sentimento e informa ao integrador se o resultado veio do modelo ou do fallback.

In [ ]:
def _analyze_churn(transcription: str, mode: str) -> tuple[dict[str, Any], str, str | None]:
    if mode == "fallback":
        return _lexical_churn(transcription), "fallback", None
    try:
        return _model_churn(transcription), "model", None
    except Exception as error:
        if mode == "full":
            raise RuntimeError("O modo full exige o modelo MiniLM NLI.") from error
        return _lexical_churn(transcription), "fallback", type(error).__name__
